# Train MFFT-Tiny on Kaggle
**Multi-Frequency Fusion Transformer — Tiny Variant (372K params)**

## Setup
1. **Settings → Accelerator**: GPU T4 x2 (or P100)
2. **Settings → Internet**: On
3. **Add Input** (attach all 11 datasets):
   - `stable-diffusion`, `places365`, `open-images-v7-dataset`
   - `ntire2026`, `midjourney`, `mfft-real`, `genimage-ai`
   - `faceforensics`, `dfdc-faces-of-the-train-sample`
   - `dall-e3`, `celebdf-v2image-dataset`
4. **Secrets → Add Secret**: `HF_TOKEN` = your HuggingFace write token

## What happens
- Downloads the frozen split manifest from HF (must exist — run `train_mfft_base` first)
- Uses the exact same train/val/test split as all other runs
- Checkpoints go to `runs/<run_id>/...`

In [ ]:
# Cell 1: Clone repo & install deps
import subprocess, sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/mfft_repo")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "https://github.com/MIHMahmudEli/ai-image-detection-research.git", str(REPO_DIR)], check=True)
    print(f"Cloned repo to {REPO_DIR}")
else:
    print(f"Repo exists at {REPO_DIR}")

sys.path.insert(0, str(REPO_DIR))

subprocess.run(["pip", "install", "-q", "huggingface_hub", "open_clip_torch", "scipy", "python-dotenv"], check=False)
print("Deps installed.")

In [ ]:
# Cell 2: Verify GPU & HF token
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "No GPU")

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    print(f"HF Token: {token[:8]}...")
except Exception as e:
    print(f"ERROR: HF_TOKEN not found as Kaggle Secret: {e}")

In [ ]:
# Cell 3: Override model variant to tiny, then run training
# We patch the config before the training script reads it
import os
os.environ["MFFT_MODEL_VARIANT"] = "tiny"

# Patch the CONFIG section in the training script
import importlib.util
script_path = str(REPO_DIR / "kaggle_train_resumable.py")

with open(script_path) as f:
    code = f.read()

# Override MODEL_VARIANT before execution
code = code.replace('MODEL_VARIANT = "base"', 'MODEL_VARIANT = os.environ.get("MFFT_MODEL_VARIANT", "tiny")')

exec(compile(code, script_path, 'exec'))

## Results

- Checkpoints: `https://huggingface.co/studyhub991/mfft-checkpoints/tree/main/runs/<run_id>`
- Run `list_active_runs.py` locally to compare with other model variants